In [ ]:
! pip install deepagents 

In [2]:
# Ollama 雲端
import os
from langchain_ollama import ChatOllama

# os.environ["OLLAMA_API_KEY"] = "這裡輸入你的API Key"

model_name = 'gemma4:31b-cloud'
llm = ChatOllama(model=model_name, base_url="https://ollama.com")

for chunk in llm.stream("一句話說明機器學習的定義"):
    print(chunk.content, end='')

機器學習是一種讓電腦無需經由明確指令編寫，而是透過**分析數據**來**發現規律**並**自我改進**能力的人工智慧技術。

In [3]:
# 匯入 geopy 套件中的 Nominatim 類別，用於地理編碼（將地名轉換成經緯度）
from geopy.geocoders import Nominatim, ArcGIS

# 定義函式：輸入城市名稱，回傳其經緯度座標
def get_coordinates(city_name):
    """取得城市GPS座標，先用 Nominatim，失敗時改用 ArcGIS"""

    # ---- 第一來源：Nominatim ----
    try:
        geolocator1 = Nominatim(user_agent="clement_fallback_test")
        location = geolocator1.geocode(city_name, timeout=10)

        if location:
            return (location.latitude, location.longitude)
    except:
        pass  # 忽略錯誤，直接進入第二來源

    # ---- 第二來源：ArcGIS  ----
    try:
        geolocator2 = ArcGIS(timeout=10)
        location = geolocator2.geocode(city_name)

        if location:
            return (location.latitude, location.longitude)
    except Exception as e:
        return e

    return None

# 範例測試：查詢「台北」的經緯度
get_coordinates("台北")

# 匯入 requests 套件，用於發送 HTTP 請求
import requests

# 定義函式：輸入經緯度，回傳目前天氣資訊（溫度）
def get_weather(latitude, longitude):
    """取得溫度值

    Args:
        latitude: GPS經度
        longitude: GPS緯度
    """    

    # 使用 Open-Meteo API 發送 GET 請求，取得氣象資料
    # API 參數：
    # - latitude / longitude: 經緯度
    # - current: 取得目前時刻的溫度 (temperature_2m) 與風速 (wind_speed_10m)
    # - hourly: 取得每小時溫度、相對濕度、風速
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}&"
        f"current=temperature_2m,wind_speed_10m&"
        f"hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    
    # 將回傳的 JSON 資料解析成 Python 字典
    data = response.json()
    
    # 回傳目前時刻的氣溫（單位：攝氏度）
    return data['current']['temperature_2m']

# 範例測試：取得台北市 (經緯度 25.0375198, 121.5636796) 的目前溫度
get_weather(25.0375198, 121.5636796)


32.6

In [7]:
# https://docs.langchain.com/oss/python/deepagents/overview

from deepagents import create_deep_agent
from langchain_core.messages import HumanMessage

agent = create_deep_agent(
    model = llm,
    tools=[get_coordinates, get_weather],
    system_prompt="",
)

messages = [
    HumanMessage("台北市現在的溫度？")
]
# Run the agent
agent.invoke({
    "messages": messages
})

{'messages': [HumanMessage(content='台北市現在的溫度？', additional_kwargs={}, response_metadata={}, id='5ec3e30d-4c96-4ff8-b1fd-015f0bc91eca'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b-cloud', 'created_at': '2026-08-16T03:12:36.662774776Z', 'done': True, 'done_reason': 'stop', 'total_duration': 534177547, 'load_duration': None, 'prompt_eval_count': 2185, 'prompt_eval_duration': None, 'eval_count': 18, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b-cloud', 'model_provider': 'ollama'}, id='lc_run--01a0088e-90f0-7a50-b44f-0d7eec91fbc9-0', tool_calls=[{'name': 'get_coordinates', 'args': {'city_name': '台北市'}, 'id': '055aa2f5-0324-4d38-91d1-66c16986ba93', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2185, 'output_tokens': 18, 'total_tokens': 2203}),
  ToolMessage(content='[25.0375198, 121.5636796]', name='get_coordinates', id='48a5714d-d02e-4eb1-a68e-75f3ea4b627c', tool_call_id='055aa2f5-0324-4d38-91d1-